# 03 · Explore — the extracted KG is a queryable graph

The knowledge graph isn't hidden behind the retriever — it's a real property graph in AgensGraph. This reuses demo 1's `lightrag_wiki` graph to show the graph-store API and a multi-hop answer.

In [1]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
# LightRAG resets its own logger to INFO on construction, so silence verbose
# INFO/WARNING logs globally (logging.disable can't be overridden by setLevel).
logging.disable(logging.WARNING)
from _common import config
from _common.rag import build_rag
from lightrag import QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
config.require_openai_key()
rag = build_rag("lightrag_wiki")
await rag.initialize_storages(); await initialize_pipeline_status()
g = rag.chunk_entity_relation_graph
print("total entities:", f"{len(await g.get_all_labels()):,}")

total entities: 15,481


## Most-connected entities (`get_popular_labels` + `node_degree`)

In [2]:
popular = await g.get_popular_labels(limit=12)
for name in popular:
    print(f"  {name:32} degree={await g.node_degree(name)}")

  Armenia                          degree=53
  United States                    degree=53
  Andhra Pradesh                   degree=49
  Abydos                           degree=42
  Angola                           degree=41
  Azerbaijan                       degree=37
  Philip Anthony Hopkins           degree=34
  Alabama                          degree=32
  Andrea Andreani                  degree=31
  Antonio Agliardi                 degree=29
  Apiaceae                         degree=29
  Arabs                            degree=29


## Label search (`search_labels`)

In [3]:
print(", ".join(await g.search_labels("Armenia", limit=10)))

Anastasius of Armenia, Armed Forces Of Armenia, Armenia, Armenia Railways, Armenia Tree Project, Armenian, Armenian Alphabet, Armenian Army, Armenian Communist Party, Armenian Diaspora


## Subgraph export — an ego-network (`get_knowledge_graph`)

In [4]:
hub = popular[0]
kg = await g.get_knowledge_graph(hub, max_depth=2, max_nodes=40)
print(f"around '{hub}': {len(kg.nodes)} nodes, {len(kg.edges)} edges, truncated={kg.is_truncated}")
for e in kg.edges[:10]:
    print(f"  ({e.source}) -[{getattr(e,'type','REL')}]- ({e.target})")

around 'Armenia': 40 nodes, 39 edges, truncated=True
  (Armenia) -[DIRECTED]- (United States)
  (Armenia) -[DIRECTED]- (Soviet Union)
  (Armenia) -[DIRECTED]- (Georgia)
  (Armenia) -[DIRECTED]- (Iran)
  (Armenia) -[DIRECTED]- (Azerbaijan)
  (Armenia) -[DIRECTED]- (Turkey)
  (Armenia) -[DIRECTED]- (Russia)
  (Armenia) -[DIRECTED]- (Council of Europe)
  (Armenia) -[DIRECTED]- (Non-Aligned Movement)
  (Armenia) -[DIRECTED]- (Nagorno-Karabakh)


## Multi-hop — connect two hubs (graph vs naive)

In [5]:
e1, e2 = popular[0], popular[1]
q = f"How are '{e1}' and '{e2}' connected? Explain any path between them."
print("Q:", q)
for mode in ["naive", "mix"]:
    ans = await rag.aquery(q, QueryParam(mode=mode, enable_rerank=False))
    print(f"\n### {mode}\n" + str(ans).strip()[:500] + " …")

Q: How are 'Armenia' and 'United States' connected? Explain any path between them.



### naive
Armenia and the United States are connected through various diplomatic, economic, and cultural ties.

### Diplomatic Relations
Armenia established diplomatic relations with the United States following its independence from the Soviet Union in 1991. The U.S. recognized Armenia on December 25, 1991, and has since been involved in supporting Armenia’s development and democracy through various programs. Armenia has sought to enhance its international presence and strengthen relations with Western na …



### mix
Armenia and the United States are connected through diplomatic relations and international cooperation, which are designed to promote positive and friendly relations. Since gaining independence, Armenia has actively sought to develop and maintain sturdy ties with the United States, reflecting its strategic interests in fostering international partnerships.

### Key Aspects of the Connection

- **Diplomatic Relations**: Armenia aims to enhance its diplomatic relations with the United States, enga …


In [6]:
await rag.finalize_storages()